# Creating an exponential integrator for the simple heat equation

In [10]:
from ngsolve import *
from ngsolve.webgui import *
import time
from ngsolve import bla
mesh = Mesh(unit_square.GenerateMesh(maxh = 0.01))
fes = H1(mesh, order=3, dirichlet = ".*")

u, v = fes.TnT()
a = BilinearForm(fes)
a +=- grad(u)*grad(v)*dx

m = BilinearForm(fes)
m += u*v*dx

a.Assemble()
m.Assemble()

f = LinearForm(fes)
f +=1* 100*sin(4*pi*(x*y-0.5))*v*dx
f.Assemble()



X = CF((x, y))
C = CF((0.5, 0.5))
u0 = 10*x*y*sin(2*pi*x)*sin(2*pi*y)*exp(-10*Norm(C-X)**2)
Draw(u0, mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

As comparision here a simple Explicit Euler:



In [11]:
gfu_expl = GridFunction(fes)
gfu_expl.Set(u0)

scene = Draw(gfu_expl)

gfu_expl.Set(u0)

dt =1e-3
tend = 1
t = 0
minv = m.mat.Inverse(fes.FreeDofs())
mstar = minv@a.mat
counter = 0
with TaskManager():
    start = time.time()
    
    while t < tend:
        gfu_expl.vec.data += dt*(minv * (a.mat*gfu_expl.vec +f.vec))
        t += dt
        counter += 1
        if counter % 100 == 0:
            scene.Redraw()
        
    end = time.time()
print(end-start)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

KeyboardInterrupt: 

In [12]:
gfu_impl = GridFunction(fes)
gfu_impl.Set(u0)

scene = Draw(gfu_impl)

gfu_impl.Set(u0)

#dt =1e-2
#tend = 1
t = 0
mstar = (m.mat - dt*a.mat).CreateSparseMatrix().Inverse(fes.FreeDofs())

counter = 0
with TaskManager():
    start = time.time()
    
    while t < tend:
        gfu_impl.vec.data = mstar*(m.mat*gfu_impl.vec +dt*1*f.vec)
        t += dt
        counter += 1
        if counter % 100 == 0:
            scene.Redraw()
        
    end = time.time()
print(end-start)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

37.75000286102295


In [13]:
Draw(gfu_impl - gfu_expl, mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [21]:
from ngsolve.eigenvalues import LOBPCG
import numpy as np
class ExpMatrix(BaseMatrix):
    def __init__(self, m, a, dt):
        self.dt = dt
        self.m = m
        self.a = a 
        self.inva = a.Inverse(fes.FreeDofs())
        self.invm = m.Inverse(fes.FreeDofs())
        self.vals, self.vecs = LOBPCG(self.a, self.m, pre = self.inva, num = 100,maxit = 3)
        self.U = BaseMatrix(self.vecs)
        self.Lambda = DiagonalMatrix(BaseVector(np.exp(-dt*self.vals)))
        super(ExpMatrix, self).__init__()
        self.EXP = (self.U @ self.Lambda @ self.U.T)




        phi_vals = []
        
        for lam in self.vals:
            z = self.dt * lam
            if abs(z) < 1e-12:
                phi_vals.append(1.0)
            else:
                phi_vals.append(np.expm1(z)/z)
        
        self.PHI = self.U @ DiagonalMatrix(BaseVector(phi_vals)) @ self.U.T







    def Mult (self, x, y):
        """
            M u_t = A u + f
            then u_t = ( M^{-1}A ) u + M^{-1}f
            Then

            if L = m^{-1}  
            The multiplication is y = exp(dt*a)*x
        """

        #y[:] = 0.0
        #print(len(x), Norm(x))

        #w1 = (self.U.T @ self.m*x).Evaluate()
        
        #print(len(w1), Norm(w1))
        #w2 = (self.Lambda * w1).Evaluate()
        
        #print(len(w2), Norm(w2))
        #w3 = (self.U * w2).Evaluate()
        
        #print(len(w3), Norm(w3))
        #y.data= w3

        y.data= self.EXP@ self.m*x

    def phi1 (self):
        #return  IdentityMatrix()   
        return  self.invm@self.a@(self.EXP-IdentityMatrix())   

    

    def Height (self):
        return self.smoother.height
    
    def Width (self):
        return self.smoother.height
    
    def CreateColVector(self):
        return self.m.CreateColVector()



In [22]:

with TaskManager():
    M = m.mat
    A = (-1*a.mat).CreateSparseMatrix()
    expm = ExpMatrix(M, A, dt)
    gfu_expon = GridFunction(fes)
    gfu_expon.Set(u0)


0 : [19.739238532152623, 49.35235960257175, 49.35347677064612, 78.98752435959733, 98.7462095358319, 98.75978962774495, 128.41781015287674, 128.54854269827013, 168.13636989679816, 168.68777202147533, 178.18706853123035, 198.09973736459546, 198.2575173940469, 248.46219937418715, 249.4075980374095, 258.60487999730725, 259.2223680273956, 290.03872048968714, 290.93994382021475, 320.55330558377887, 341.2166591628683, 343.47089124201244, 371.86820564532053, 379.3376835568755, 405.5191088053287, 408.3772083267991, 417.15876819001863, 422.36933812322263, 458.5339695148017, 460.59264325198455, 509.9760156400567, 522.0543949038621, 523.7083793703097, 542.3298484905016, 544.3203382644369, 551.8698621476725, 571.9045352894468, 611.9377960811054, 625.852419443254, 643.349365645347, 657.8140515813864, 685.6533350122495, 703.3283331046722, 707.7220972080643, 725.482432459009, 755.0564374268527, 778.1483979579484, 799.8984636978178, 808.3146819893916, 816.6061014781143, 842.7621644323275, 855.602997944

In [23]:
gfu_expon.Set(u0)

scene = Draw(gfu_expon)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [26]:
gfu_expon.Set(u0)
counter = 0
tend = 1
t = 0
#dt =1e-2
with TaskManager():
    start = time.time()
    
    while t < tend:
        #gfu_expon.vec.data =expm*gfu_expon.vec + dt* expm.invm *f.vec
        gfu_expon.vec.data = expm*gfu_expon.vec + dt* expm.PHI * f.vec

        t += dt
        counter += 1
        print(t, end = "\r")
        if counter % 10 == 0:
            scene.Redraw()
        
    end = time.time()
print(end-start)





16.80676579475403776


In [27]:
Draw(Norm(gfu_expon- gfu_expl), mesh)
Draw(Norm(gfu_expon- gfu_impl), mesh)
Draw(Norm(gfu_expl-  gfu_impl), mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene